In [1]:
import sys, json, os
sys.path.append("../src")

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import joblib

from features import RAW_FEATURES
from metrics import evaluate
from reference import build_reference
from drift import feature_psi, score_psi
from monitor import GROUPS, group_psi, check_rules_v2
from retrain import train_challenger, window, matured_end, TRAIN_DAYS, HOLDOUT_DAYS

feat = pd.read_parquet("../data/processed/features.parquet")

if os.path.exists("../models/c7.joblib"):
    m7 = joblib.load("../models/c7.joblib")
else:
    m7, _, _ = train_challenger(feat, 7)

me = matured_end(7)
tr7 = window(feat, me - HOLDOUT_DAYS - TRAIN_DAYS, me - HOLDOUT_DAYS)
ho7 = window(feat, me - HOLDOUT_DAYS, me)

ho_scores = m7.predict_proba(ho7[RAW_FEATURES])[:, 1]
imp = pd.Series(m7.booster_.feature_importance(importance_type="gain"), index=RAW_FEATURES)
ref7 = build_reference(tr7[RAW_FEATURES], RAW_FEATURES, ho_scores, imp)
baseline7 = evaluate(ho7["isFraud"], ho_scores)["pr_auc"]

print("c7 holdout PR-AUC at deployment:", baseline7)
print("Performance alert line (20% below):", round(0.8 * baseline7, 3))

late = feat[(feat.split == "prod") & (feat.batch >= 8)].copy()
print(late.batch.value_counts().sort_index().to_dict())

c7 holdout PR-AUC at deployment: 0.6351
Performance alert line (20% below): 0.508
{8: 20263, 9: 20967, 10: 20802, 11: 20389}


In [2]:
def clean(df):
    return df.copy()

def amount_x15(df):
    """Prices suddenly 50% higher."""
    d = df.copy()
    d["TransactionAmt"] = d["TransactionAmt"] * 1.5
    return d

def feed_outage(df):
    """A data feed breaks: all M and D columns stop arriving."""
    d = df.copy()
    for c in GROUPS["D"]:
        d[c] = np.float32(np.nan)
    for c in GROUPS["M"]:
        d[c] = pd.Categorical([None] * len(d), categories=df[c].cat.categories)
    return d

def concept_shift(df, seed=0):
    """Half of the fraud labels stop following the old patterns (inputs unchanged)."""
    d = df.copy()
    rng = np.random.default_rng(seed)
    rate = d["isFraud"].mean()
    swap = rng.random(len(d)) < 0.5
    d["isFraud"] = np.where(swap, rng.random(len(d)) < rate, d["isFraud"]).astype(int)
    return d

In [3]:
DEPLOYED = 7

def signals(df, scenario):
    d = scenario(df)
    d["s"] = m7.predict_proba(d[RAW_FEATURES])[:, 1]
    rows = []
    for w in range(8, 12):
        b = d[d.batch == w]
        fp = feature_psi(b[RAW_FEATURES], ref7)
        gp = group_psi(fp)
        sc = score_psi(b["s"].to_numpy(), ref7)

        pooled = None
        weeks_ok = list(range(DEPLOYED + 1, w - 1))
        if len(weeks_ok) >= 2:
            dd = d[d.batch.isin(weeks_ok[-2:])]
            pooled = evaluate(dd["isFraud"], dd["s"])["pr_auc"]

        reasons = check_rules_v2(fp, sc, pooled, baseline7)
        rows.append({"week": w, "score_psi": round(sc, 3),
                     "groups_over": sum(v > 0.25 for v in gp.values()),
                     "worst_group": max(gp, key=gp.get), "worst_psi": round(max(gp.values()), 2),
                     "pooled_pr_auc": None if pooled is None else round(pooled, 3),
                     "alert": ", ".join(reasons) or "-"})
    return pd.DataFrame(rows)


scenarios = {
    "clean (no injection)": clean,
    "amount x1.5": amount_x15,
    "feed outage (M and D blank)": feed_outage,
    "concept shift (half the labels random)": concept_shift,
}

stress = {}
for name, fn in scenarios.items():
    stress[name] = signals(late, fn)
    print("\n" + name)
    print(stress[name].to_string(index=False))


clean (no injection)
 week  score_psi  groups_over worst_group  worst_psi  pooled_pr_auc            alert
    8      0.001            0           D       0.12            NaN                -
    9      0.001            0           D       0.16            NaN                -
   10      0.014            0       email       0.07            NaN                -
   11      0.030            0           D       0.10          0.505 performance drop

amount x1.5
 week  score_psi  groups_over    worst_group  worst_psi  pooled_pr_auc            alert
    8      0.015            1 amount_product       0.57            NaN                -
    9      0.016            1 amount_product       0.55            NaN                -
   10      0.053            1 amount_product       0.52            NaN                -
   11      0.065            1 amount_product       0.51          0.493 performance drop

feed outage (M and D blank)
 week  score_psi  groups_over worst_group  worst_psi  pooled_pr_auc    

Stress test of rule v2 on weeks 8-11 (model c7 serving, rules frozen). A feed outage (all M and D columns blank) was caught in the first affected week by the score-drift and feature-drift rules (score PSI 0.52-0.68). A concept shift (half of the labels replaced by random draws, inputs unchanged) was invisible to the score and feature checks, and only the performance rule fired, at week 11 (pooled PR-AUC 0.163), three weeks after onset because of the label delay. Amount ×1.5 was not caught by the drift rules, because only one feature group was affected (PSI 0.51-0.57) and the rule requires two. On untouched data the performance rule fired at week 11 (pooled PR-AUC 0.505 vs line 0.508), so it is not specific. Each scenario gives the performance rule one decision, so this is a small smoke test and not a validation.

In [4]:
out = pd.concat(stress, names=["scenario"]).reset_index(level=0)
out.to_csv("../docs/day7_stress_test.csv", index=False)

summ = []
for name, df_ in stress.items():
    fired = df_[df_.alert != "-"]
    summ.append({"scenario": name,
                 "alert weeks": list(fired.week),
                 "rules that fired": "; ".join(sorted(set(", ".join(fired.alert).split(", ")))) if len(fired) else "-"})
pd.DataFrame(summ)

,scenario,alert weeks,rules that fired
0,clean (no injection),[11],performance drop
1,amount x1.5,[11],performance drop
2,feed outage (M and D blank),"[8, 9, 10, 11]",feature drift; performance drop; score drift
3,concept shift (half the labels random),[11],performance drop


In [5]:
import pandas as pd
weekly = pd.read_csv("../docs/day6_weekly_by_policy.csv")
cum = weekly.sort_values(["policy", "week"]).copy()
cum["cumulative"] = cum.groupby("policy")["net_benefit"].cumsum()
piv = cum.pivot(index="week", columns="policy", values="cumulative")

extra = piv[["every 4 weeks", "on alert"]].sub(piv["never retrain"], axis=0)
step = extra.diff().fillna(extra)
print(step.round(0))
print("\nWeek 9 share of total extra:")
print((step.loc[9] / extra.loc[11]).round(2))

policy  every 4 weeks  on alert
week                           
0                 0.0       0.0
1                 0.0   -3722.0
2                 0.0   -2638.0
3                 0.0    1469.0
4              5677.0   -1454.0
5             -1116.0    1162.0
6              8540.0   10583.0
7              3119.0    -767.0
8              6322.0    6322.0
9             20063.0   20063.0
10             7972.0    7972.0
11             2251.0    2251.0

Week 9 share of total extra:
policy
every 4 weeks    0.38
on alert         0.49
dtype: float64


In [6]:
import json
import pandas as pd

events = pd.read_csv("../docs/day6_events.csv")
ev = events[(events.policy == "on alert") & (events.promoted == True)].sort_values("week")

old = json.load(open("../models/registry.json"))
v1 = next(r for r in old if r["version"] == "v1")        # keep the original v1 record

new = [dict(v1, status="champion (simulated)")]
for _, e in ev.iterrows():
    w = int(e["week"])
    me = 90 + 7 * (w + 1) - 14                            # last matured day at the end of week w
    new[-1]["status"] = f"retired (end of week {w})"
    new.append({"version": e["challenger"],
                "promoted_end_of_week": w,
                "trained_on_days": [me - 74, me - 14],
                "threshold": round(float(e["chall_threshold"]), 4),
                "holdout_pr_auc": round(float(e["chall_holdout_pr_auc"]), 4),
                "reasons": e["reasons"],
                "status": "champion (simulated)"})

json.dump(new, open("../models/registry.json", "w"), indent=2)
print([(r["version"], r["status"]) for r in new])

[('v1', 'retired (end of week 0)'), ('c0', 'retired (end of week 2)'), ('c2', 'retired (end of week 4)'), ('c4', 'retired (end of week 7)'), ('c7', 'champion (simulated)')]
